# Sparkov M4 — Isolation Forest Anomaly Score + M3 LightGBM

## 1. Objective and methodology

M4 extends the **selected Sparkov M3 LightGBM** with a single Isolation Forest anomaly-score feature. This follows the **feature-level integration** approach in IEEE-CIS M4 notebook, but retains the Sparkov M3 pseudo-labelling strategy because the base-model choice is dataset-specific.

- **M3 retained:** rank-based fraud/legitimate pseudo-label selection, class-specific reliability multipliers, class balancing derived from verified labels, weekly expanding-window retraining, and seven-day label verification delay.
- **Anomaly model:** one Isolation Forest, fitted at each weekly cutoff using transactions observed strictly before that cutoff; labels are not used in anomaly fitting. Its training sample is capped at 100,000 rows for efficiency.
- **Integration:** the anomaly score is an extra numeric LightGBM feature. Higher values indicate greater anomaly. The verified-only model that selects pseudo-labels remains the same as in M3, so the pseudo-label selection procedure is held fixed.
- **Evaluation:** one prespecified Isolation Forest configuration on held-out chronological validation, compared with the previously saved M3 validation predictions. **No test-set evaluation or operational alert-threshold tuning is performed here.**

This is a controlled, limited M4 experiment rather than IEEE's full multi-variant search. Different anomaly variants and hyperparameter searches are intentionally omitted to limit computational cost; the limitation should be disclosed in the report.


## 2. Load Sparkov data and reproduce M3 infrastructure

The following functions are carried over from the completed Sparkov M3 notebook. Do **not** run the M3 development grid again.

In [4]:
from pathlib import Path
from itertools import product
import time
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/m4_sparkov_iforest")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the processed feature table and the pre-approved modelling feature list.
df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")
with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)
feature_cols = feature_lists["sparkov_feature_cols"]

TIME_COL = "trans_date_trans_time"
TARGET_COL = "is_fraud"
SPLIT_COL = "sparkov_split"

# The test partition is excluded. Retain chronological train and validation only.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
m3_df = df.loc[df[SPLIT_COL].isin(["train", "validation"])].copy()
m3_df = m3_df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
del df

# Match M2: reconstruct selected frequency features using strictly earlier rows.
frequency_sources = {
    "merchant_freq": "merchant",
    "category_freq": "category",
    "state_freq": "state",
}
for feature, source in frequency_sources.items():
    if feature not in feature_cols:
        continue
    previous_rows = m3_df.groupby(source, dropna=False).cumcount()
    same_timestamp_previous = m3_df.groupby(
        [source, TIME_COL], dropna=False
    ).cumcount()
    m3_df[feature] = (previous_rows - same_timestamp_previous).astype("int64")

train_mask = m3_df[SPLIT_COL].eq("train")
X_train_schema = m3_df.loc[train_mask, feature_cols]
categorical_cols = X_train_schema.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Data shape:", m3_df.shape)
print("Features:", len(feature_cols))
print("Categorical features:", categorical_cols)
print("Fraud rates by split:")
display(m3_df.groupby(SPLIT_COL)[TARGET_COL].agg(["size", "mean"]))


Data shape: (1296675, 87)
Features: 62
Categorical features: ['category', 'gender', 'state']
Fraud rates by split:


,size,mean
sparkov_split,,
train,1037340,0.005753
validation,259335,0.005931


In [5]:
# Fixed M2 temporal configuration
WINDOW_DAYS = None                 # expanding history
RETRAIN_EVERY_DAYS = 7
LABEL_DELAY_DAYS = 7
# Compact, prespecified 4-parameter grid (16 configurations).
# Lycia's selection rule and class-specific reliability weights are retained.
FRAUD_THRESHOLDS = [0.15, 0.50]
LEGIT_THRESHOLDS = [0.25, 0.50]
FRAUD_RELIABILITIES = [0.15, 0.50]
LEGIT_RELIABILITIES = [0.30, 0.75]


def make_lightgbm(class_weight="balanced"):
    """Best LightGBM tree settings from M1/M2."""
    return lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        objective="binary",
        class_weight=class_weight,
        is_unbalance=False,
        random_state=100,
        n_jobs=2,
        verbosity=-1,
    )


def prepare_features(frame, categories=None):
    X = frame[feature_cols].copy()
    for col in categorical_cols:
        if categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=categories[col])
    return X


def verified_training_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select verified labels available at this cutoff; use expanding history."""
    cutoff = pd.Timestamp(cutoff)
    latest_eligible = cutoff - pd.Timedelta(days=label_delay_days)
    mask = frame[SPLIT_COL].isin(allowed_splits) & frame[TIME_COL].lt(latest_eligible)
    return frame.loc[mask].copy()


def pseudo_candidate_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select rows in the seven-day label-delay interval before the cutoff."""
    cutoff = pd.Timestamp(cutoff)
    start = cutoff - pd.Timedelta(days=label_delay_days)
    mask = (
        frame[SPLIT_COL].isin(allowed_splits)
        & frame[TIME_COL].ge(start)
        & frame[TIME_COL].lt(cutoff)
    )
    return frame.loc[mask].copy()


def make_cutoffs(start, end, frequency_days=RETRAIN_EVERY_DAYS):
    return pd.date_range(
        start=pd.Timestamp(start),
        end=pd.Timestamp(end),
        freq=f"{frequency_days}D",
        inclusive="left",
    )


def precompute_pseudo_scores(
    frame,
    start,
    end,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Cache one verified-only model's scores for each cutoff's delay window."""
    cache = {}
    cutoffs = make_cutoffs(start, end, frequency_days)

    for i, cutoff in enumerate(cutoffs, start=1):
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        candidates = pseudo_candidate_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        if verified.empty or verified[TARGET_COL].nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")
        if candidates.empty:
            cache[cutoff] = pd.DataFrame(
                columns=["row_index", "fraud_probability"]
            )
            continue

        X_verified = prepare_features(verified)
        categories = {col: X_verified[col].cat.categories for col in categorical_cols}
        X_candidates = prepare_features(candidates, categories=categories)
        model = make_lightgbm(class_weight="balanced")
        model.fit(X_verified, verified[TARGET_COL].astype(int))
        fraud_probability = model.predict_proba(X_candidates)[:, 1]

        cache[cutoff] = pd.DataFrame({
            "row_index": candidates.index.to_numpy(),
            "fraud_probability": fraud_probability,
        })
        print(
            f"Cached {i}/{len(cutoffs)} cutoffs: {cutoff} "
            f"({len(verified):,} verified; {len(candidates):,} unlabeled candidates)",
            flush=True,
        )
    return cache


def balanced_class_weight_map(y_verified):
    """Balanced class weights based only on verified labels at this cutoff."""
    classes = np.sort(pd.Series(y_verified).astype(int).unique())
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=pd.Series(y_verified).astype(int).to_numpy(),
    )
    return {int(label): float(weight) for label, weight in zip(classes, weights)}


def summarize_predictions(predictions):
    y_true = predictions["y_true"].astype(int)
    y_score = predictions["y_score"].astype(float)
    return {
        "transactions": len(predictions),
        "fraud_cases": int(y_true.sum()),
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score),
    }


## 3. M3 pseudo-label selection (unchanged)

For each weekly cutoff, a verified-only M2 model scores the most recent seven days of unverified transactions. The selected M3 thresholds and reliability multipliers are read from the saved M3 configuration, **not retuned**. We use the exact same rank-based selection and weighting logic as Notebook 6.

In [6]:
M3_RESULTS_DIR = Path("../results/m3_rank_aligned_ieee")
config_path = M3_RESULTS_DIR / "m3_best_config.csv"
if not config_path.exists():
    raise FileNotFoundError(f"Missing M3 config: {config_path}. Keep Notebook 6 results accessible.")
best_config = pd.read_csv(config_path).iloc[0].to_dict()
M3_PARAMS = {key: float(best_config[key]) for key in
             ["fraud_thres", "legit_thres", "fraud_reliability", "legit_reliability"]}
print("Fixed M3 parameters:", M3_PARAMS)
assert np.isclose(M3_PARAMS["fraud_thres"], .5)
assert np.isclose(M3_PARAMS["legit_thres"], .5)
assert np.isclose(M3_PARAMS["fraud_reliability"], .5)
assert np.isclose(M3_PARAMS["legit_reliability"], .3)


Fixed M3 parameters: {'fraud_thres': 0.5, 'legit_thres': 0.5, 'fraud_reliability': 0.5, 'legit_reliability': 0.3}


## 4. Isolation Forest anomaly score

Isolation Forest uses numeric, model-ready predictor columns only. It excludes the target, split, timestamp and identifiers by drawing strictly from the approved `feature_cols` list. Missing and infinite values are replaced using medians calculated **only from the historical fitting sample**. Anomaly scores are the negative of `score_samples`, so larger scores correspond to more anomalous transactions.

The forest is fitted afresh for each cutoff using up to 100,000 historical transactions observed before the cutoff. For all rows scored at a cutoff (verified, pseudo-labelled and next-week prediction rows), **the same forest and imputation statistics** are used. It never fits on the upcoming prediction block.

In [7]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer

IFOREST_SETTINGS = dict(n_estimators=200, max_samples=256, random_state=100, n_jobs=2)
IFOREST_FIT_CAP = 100_000
IFOREST_SCORE_COL = "iforest_anomaly_score"

# Numeric feature selection is derived from the approved model features only.
ANOMALY_COLS = [c for c in feature_cols if pd.api.types.is_numeric_dtype(m3_df[c])]
if not ANOMALY_COLS:
    raise ValueError("No numeric anomaly-detection features found")
print("Isolation Forest numeric predictors:", len(ANOMALY_COLS))


def build_anomaly_feature(frame, cutoff, fit_rows, score_rows, pseudo_rows):
    """Fit an unsupervised forest on strictly prior observed rows; score all needed rows."""
    historical = frame.loc[frame[TIME_COL].lt(pd.Timestamp(cutoff))]
    if historical.empty:
        raise ValueError(f"No historical transactions before {cutoff}")
    if len(historical) > IFOREST_FIT_CAP:
        historical = historical.sample(n=IFOREST_FIT_CAP, random_state=100)
    # Convert infinities to missing; learn imputation only from historical data.
    hist_x = historical[ANOMALY_COLS].replace([np.inf, -np.inf], np.nan)
    # Median-impute columns with at least one observed value in the fitting sample.
    usable = hist_x.columns[hist_x.notna().any(axis=0)].tolist()
    if not usable:
        raise ValueError("No usable historical anomaly predictors")
    imputer = SimpleImputer(strategy="median")
    x_hist = imputer.fit_transform(hist_x[usable]).astype("float32")
    forest = IsolationForest(**IFOREST_SETTINGS)
    forest.fit(x_hist)

    def transform_score(rows):
        if rows.empty:
            return np.empty(0, dtype=float)
        x = rows[usable].replace([np.inf, -np.inf], np.nan)
        x = imputer.transform(x).astype("float32")
        return -forest.score_samples(x)

    return transform_score(fit_rows), transform_score(score_rows), len(historical)


Isolation Forest numeric predictors: 59


## 5. Chronological M4 validation pipeline

This is the Notebook 6 M3 training procedure with exactly one additional feature. The seven-day label delay is enforced for **verified labels**; the Isolation Forest may use already observed unlabelled transactions, since its fitting does not use fraud outcomes.

**Note:** The forest is fitted on past observed transactions, including those whose labels may not yet be verified. This is a transductive use of observed, unlabelled history—not future transactions.

In [8]:
def run_m4_lightgbm(
    frame,
    start,
    end,
    fraud_thres=None,
    legit_thres=None,
    fraud_reliability=0.0,
    legit_reliability=0.0,
    pseudo_score_cache=None,
    allowed_splits=("train",),
    scoring_split="train",
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Weekly M3 LightGBM retraining with an Isolation Forest feature."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start >= end:
        raise ValueError("start must be earlier than end")
    use_pseudo = fraud_thres is not None
    if not use_pseudo:
        raise ValueError("M4 must extend the selected M3, not verified-only M2")
    if use_pseudo:
        if legit_thres is None:
            raise ValueError("legit_thres is required for M3")
        if not 0.0 < fraud_thres <= 1.5 or not 0.0 < legit_thres < 1.0:
            raise ValueError("fraud_thres must be in (0, 1.5], legit_thres in (0, 1)")
        if not 0 <= fraud_reliability <= 1 or not 0 <= legit_reliability <= 1:
            raise ValueError("Reliability weights must be in [0, 1]")

    prediction_blocks = []
    for cutoff in make_cutoffs(start, end, frequency_days):
        next_cutoff = min(cutoff + pd.Timedelta(days=frequency_days), end)
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        score_block = frame.loc[
            frame[SPLIT_COL].eq(scoring_split)
            & frame[TIME_COL].ge(cutoff)
            & frame[TIME_COL].lt(next_cutoff)
        ].copy()
        if score_block.empty:
            continue
        y_verified = verified[TARGET_COL].astype(int)
        if verified.empty or y_verified.nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")

        pseudo_rows = frame.iloc[0:0].copy()
        pseudo_rows["_m3_target"] = pd.Series(dtype=int)
        pseudo_rows["_m3_reliability"] = pd.Series(dtype=float)
        if use_pseudo:
            if pseudo_score_cache is None or cutoff not in pseudo_score_cache:
                raise KeyError(f"Missing pseudo-label scores for {cutoff}")
            scored_candidates = pseudo_score_cache[cutoff]
            if not scored_candidates.empty:
                # Same rank-based top/bottom selection as Lycia's IEEE M3.
                p = scored_candidates["fraud_probability"].to_numpy()
                sorted_scores = np.sort(p)
                verified_rate = float(y_verified.mean())
                n_fraud = min(max(int(round(fraud_thres * verified_rate * len(p))), 1), len(p))
                n_legit = min(max(int(round(legit_thres * len(p))), 1), len(p))
                fraud_cutoff = sorted_scores[-n_fraud]
                legit_cutoff = sorted_scores[n_legit - 1]
                is_fraud = p >= fraud_cutoff
                is_legit = p <= legit_cutoff
                # Lycia's implementation gives fraud precedence when masks overlap.
                keep = is_fraud | is_legit
                chosen = scored_candidates.loc[keep]
                if not chosen.empty:
                    pseudo_rows = frame.loc[chosen["row_index"].astype(int).to_numpy()].copy()
                    pseudo_rows["_m3_target"] = is_fraud[keep].astype(int)
                    pseudo_rows["_m3_reliability"] = np.where(
                        is_fraud[keep], fraud_reliability, legit_reliability
                    )

        verified_rows = verified.copy()
        verified_rows["_m3_target"] = y_verified.to_numpy()
        fit_rows = pd.concat([verified_rows, pseudo_rows], axis=0)
        y_fit = fit_rows["_m3_target"].astype(int)

        # Per-row sample weight = class weight from verified labels × reliability.
        class_weights = balanced_class_weight_map(y_verified)
        class_component = y_fit.map(class_weights).to_numpy(dtype=float)
        reliability_component = np.concatenate([
            np.ones(len(verified_rows), dtype=float),
            pseudo_rows["_m3_reliability"].to_numpy(dtype=float),
        ])
        fit_sample_weight = class_component * reliability_component

        anomaly_fit, anomaly_score, iforest_fit_rows = build_anomaly_feature(
            frame, cutoff, fit_rows, score_block, pseudo_rows
        )
        X_fit = prepare_features(fit_rows)
        X_fit[IFOREST_SCORE_COL] = anomaly_fit
        categories = {col: X_fit[col].cat.categories for col in categorical_cols}
        X_score = prepare_features(score_block, categories=categories)
        X_score[IFOREST_SCORE_COL] = anomaly_score
        # Class balance is already represented in fit_sample_weight, avoiding double weighting.
        model = make_lightgbm(class_weight=None)
        model.fit(X_fit, y_fit, sample_weight=fit_sample_weight)
        scores = model.predict_proba(X_score)[:, 1]

        prediction_blocks.append(pd.DataFrame({
            "row_index": score_block.index.to_numpy(),
            "timestamp": score_block[TIME_COL].to_numpy(),
            "y_true": score_block[TARGET_COL].astype(int).to_numpy(),
            "y_score": scores,
            "iforest_score": anomaly_score,
            "cutoff": cutoff,
            "fraud_thres": fraud_thres,
            "legit_thres": legit_thres,
            "fraud_reliability": fraud_reliability,
            "legit_reliability": legit_reliability,
            "verified_rows": len(verified_rows),
            "pseudo_rows": len(pseudo_rows),
            "iforest_fit_rows": iforest_fit_rows,
            "class_weight_legitimate": class_weights[0],
            "class_weight_fraud": class_weights[1],
        }))

    if not prediction_blocks:
        raise ValueError("No scoring rows found")
    predictions = pd.concat(prediction_blocks, ignore_index=True)
    if predictions["row_index"].duplicated().any():
        raise AssertionError("Duplicate scored rows")
    return predictions


## 6. Held-out validation: M3 vs M4

Read Notebook 6's saved M3 predictions to avoid retraining M3. Precompute the same verified-only pseudo-label probabilities used by M3, then run **one** M4 validation backtest. The M3 and M4 predictions must cover identical validation transactions, with matching timestamps and true labels.

**Expected runtime:** This can still be substantial: each weekly cutoff requires a verified-only LightGBM for pseudo-label scoring, an Isolation Forest, and an M4 LightGBM. Do not interrupt the kernel or rerun completed cells unnecessarily.

In [9]:
validation_rows = m3_df.loc[m3_df[SPLIT_COL].eq("validation")]
VAL_START = validation_rows[TIME_COL].min()
VAL_END = validation_rows[TIME_COL].max().floor("D") + pd.Timedelta(days=1)
M3_PRED_PATH = M3_RESULTS_DIR / "m3_validation_predictions.csv"
if not M3_PRED_PATH.exists():
    raise FileNotFoundError(f"Missing M3 validation predictions: {M3_PRED_PATH}")
m3_validation_predictions = pd.read_csv(M3_PRED_PATH, parse_dates=["timestamp", "cutoff"])
expected_rows = set(m3_df.index[m3_df[SPLIT_COL].eq("validation")])
assert len(m3_validation_predictions) == len(expected_rows)
assert set(m3_validation_predictions["row_index"].astype(int)) == expected_rows
m3_metrics = summarize_predictions(m3_validation_predictions)
print("Saved M3 validation PR-AUC:", round(m3_metrics["pr_auc"], 6))
assert np.isclose(m3_metrics["pr_auc"], 0.985322, atol=1e-5, rtol=0)

# The M3 grid search is NOT rerun. These verified-only scores are required
# to reproduce M3's pseudo-label selection at each weekly cutoff.
VAL_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df, start=VAL_START, end=VAL_END,
    allowed_splits=("train", "validation"),
)


Saved M3 validation PR-AUC: 0.985322
Cached 1/16 cutoffs: 2020-03-06 07:16:43 (1,023,767 verified; 13,573 unlabeled candidates)
Cached 2/16 cutoffs: 2020-03-13 07:16:43 (1,037,340 verified; 16,114 unlabeled candidates)
Cached 3/16 cutoffs: 2020-03-20 07:16:43 (1,053,454 verified; 16,087 unlabeled candidates)
Cached 4/16 cutoffs: 2020-03-27 07:16:43 (1,069,541 verified; 15,950 unlabeled candidates)
Cached 5/16 cutoffs: 2020-04-03 07:16:43 (1,085,491 verified; 15,855 unlabeled candidates)
Cached 6/16 cutoffs: 2020-04-10 07:16:43 (1,101,346 verified; 16,057 unlabeled candidates)
Cached 7/16 cutoffs: 2020-04-17 07:16:43 (1,117,403 verified; 15,984 unlabeled candidates)
Cached 8/16 cutoffs: 2020-04-24 07:16:43 (1,133,387 verified; 15,766 unlabeled candidates)
Cached 9/16 cutoffs: 2020-05-01 07:16:43 (1,149,153 verified; 15,872 unlabeled candidates)
Cached 10/16 cutoffs: 2020-05-08 07:16:43 (1,165,025 verified; 15,677 unlabeled candidates)
Cached 11/16 cutoffs: 2020-05-15 07:16:43 (1,180,702

In [11]:
m4_validation_predictions = run_m4_lightgbm(
    frame=m3_df, start=VAL_START, end=VAL_END,
    **M3_PARAMS, pseudo_score_cache=VAL_PSEUDO_CACHE,
    allowed_splits=("train", "validation"),
    scoring_split="validation",
)

assert len(m4_validation_predictions) == len(expected_rows)
assert set(m4_validation_predictions["row_index"].astype(int)) == expected_rows

comparison = m3_validation_predictions[["row_index", "timestamp", "y_true", "y_score"]].merge(
    m4_validation_predictions[["row_index", "timestamp", "y_true", "y_score"]],
    on="row_index", suffixes=("_m3", "_m4"), validate="one_to_one"
)
assert (comparison["y_true_m3"] == comparison["y_true_m4"]).all()
assert (pd.to_datetime(comparison["timestamp_m3"]) == pd.to_datetime(comparison["timestamp_m4"])).all()

m4_metrics = summarize_predictions(m4_validation_predictions)
results = pd.DataFrame([
    {"model": "M3 LightGBM + pseudo-labels", **m3_metrics},
    {"model": "M4 LightGBM + pseudo-labels + Isolation Forest", **m4_metrics},
])
display(results)
print("M4 minus M3 PR-AUC:", round(m4_metrics["pr_auc"] - m3_metrics["pr_auc"], 6))

results.to_csv(RESULTS_DIR / "m4_validation_comparison.csv", index=False)
m4_validation_predictions.to_csv(RESULTS_DIR / "m4_validation_predictions.csv", index=False)
print("Saved M4 results to", RESULTS_DIR)


,model,transactions,fraud_cases,pr_auc,roc_auc
0,M3 LightGBM + pseudo-labels,259335,1538,0.985322,0.999523
1,M4 LightGBM + pseudo-labels + Isolation Forest,259335,1538,0.983973,0.999485


M4 minus M3 PR-AUC: -0.00135
Saved M4 results to ..\results\m4_sparkov_iforest



## 7. Results, Interpretation and Next Steps

### Held-Out Validation Results

M4 was evaluated against the selected M3 LightGBM model on the same chronological validation set, comprising **259,335 transactions and 1,538 fraudulent transactions**.

| Metric | M3 LightGBM | M4 LightGBM + Isolation Forest |
|---|---:|---:|
| PR-AUC | **0.985322** | 0.983973 |
| ROC-AUC | **0.999523** | 0.999485 |

Adding the Isolation Forest anomaly-score feature reduced validation PR-AUC by **0.001349** and ROC-AUC by **0.000038** relative to M3.

### Interpretation

The results indicate that the additional anomaly-score feature did not improve predictive discrimination on the Sparkov dataset under the evaluated configuration. Despite incorporating unsupervised information about unusual transaction patterns, M4 performed slightly worse than M3 on both ranking metrics.

One possible explanation is that the existing LightGBM features already capture much of the information represented by the anomaly score. However, this explanation was not directly tested and should be treated as a hypothesis rather than an established finding.

Moreover, M4 introduces additional computational overhead because Isolation Forest must be fitted and its anomaly scores generated at each weekly retraining cutoff. Given the observed deterioration in validation performance, this added complexity is not justified by the current results.

### Limitations

Only one prespecified Isolation Forest configuration was evaluated, using feature-level integration. Unlike the IEEE-CIS experiments, alternative anomaly detectors and multiple anomaly-score configurations were not explored. This narrower experimental scope reflects computational constraints and does not establish that other anomaly-detection approaches would be ineffective.

Furthermore, Sparkov is a synthetic dataset, and its already high baseline discrimination performance may limit the measurable incremental benefits of additional modelling techniques.

### Model Selection and Next Steps

| Model | Validation PR-AUC |
|---|---:|
| M2 LightGBM | 0.985243 |
| **M3 LightGBM** | **0.985322** |
| M4 LightGBM + Isolation Forest | 0.983973 |

M3 achieved the highest validation PR-AUC, although its improvement over M2 was negligible. M4 did not provide further improvement.

Based on the primary validation metric, **M3 is the leading candidate for final Sparkov evaluation**. Its marginal advantage over M2 should nevertheless be interpreted cautiously, particularly when considering the additional complexity of pseudo-labelling.

The next stage is to select the operational daily top-k fraud-alert threshold using validation predictions, followed by final evaluation on the untouched test set. The selected alert threshold will be fixed before test evaluation to avoid test-set-driven optimisation.




## 8. Standalone Isolation Forest Evaluation

To assess the independent predictive value of Isolation Forest, we evaluate its anomaly scores directly against the true fraud labels on the held-out validation set.

Unlike M4, which incorporates the anomaly score as an additional LightGBM feature, this evaluation measures Isolation Forest's standalone fraud-ranking performance.

We report PR-AUC and ROC-AUC and compare these results with M3 and M4. This helps determine whether the anomaly score provides useful fraud-discrimination information independently of the supervised classifier.


In [12]:

# 8. Standalone Isolation Forest Evaluation

from sklearn.metrics import average_precision_score, roc_auc_score

assert "iforest_score" in m4_validation_predictions.columns, (
    "Missing iforest_score. Rerun the updated M4 validation cell."
)

y_true = m4_validation_predictions["y_true"]
iforest_scores = m4_validation_predictions["iforest_score"]

iforest_pr_auc = average_precision_score(y_true, iforest_scores)
iforest_roc_auc = roc_auc_score(y_true, iforest_scores)

print(f"Isolation Forest standalone PR-AUC: {iforest_pr_auc:.6f}")
print(f"Isolation Forest standalone ROC-AUC: {iforest_roc_auc:.6f}")
print(f"Validation fraud prevalence: {y_true.mean():.6f}")


Isolation Forest standalone PR-AUC: 0.610754
Isolation Forest standalone ROC-AUC: 0.971032
Validation fraud prevalence: 0.005931



## 7. Additional M3 Classification Metrics

To complement PR-AUC, we evaluate precision and recall using a fixed fraud-probability cutoff of 0.5, alongside Precision@100 per day.

Precision@100 measures the proportion of fraudulent transactions among the 100 highest-scoring transactions each day. These metrics are calculated separately on validation and test using saved M3 predictions, without retraining or additional threshold optimisation.


In [13]:

from sklearn.metrics import precision_score, recall_score

def additional_m3_metrics(pred, split):
    y_true = pred["y_true"].astype(int)
    y_score = pred["y_score"].astype(float)

    # Fixed probability threshold of 0.5
    y_hat = (y_score >= 0.5).astype(int)

    # Top 100 transactions per day
    top100_flags = daily_topk_flags(pred, 100)

    return {
        "split": split,
        "precision_at_0.5": precision_score(
            y_true, y_hat, zero_division=0
        ),
        "recall_at_0.5": recall_score(
            y_true, y_hat, zero_division=0
        ),
        "precision_at_100_per_day": precision_score(
            y_true, top100_flags.astype(int), zero_division=0
        ),
        "alerts_at_0.5": int(y_hat.sum()),
        "alerts_at_100_per_day": int(top100_flags.sum()),
    }

additional_metrics = pd.DataFrame([
    additional_m3_metrics(val_pred, "validation"),
    additional_m3_metrics(test_pred, "test"),
])

display(additional_metrics.round(6))

additional_metrics.to_csv(
    RESULTS_DIR / "sparkov_m3_additional_metrics.csv",
    index=False
)


NameError: name 'val_pred' is not defined